In [ ]:
!pip install --upgrade transformers sentencepiece -q

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
DATA_DIR = '/content/'
OUTPUT_DIR = '/content/retrieval'

from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
!pip install faiss-cpu

In [ ]:
import json
import numpy as np
from typing import List, Dict, Tuple
from tqdm.auto import tqdm
from dataclasses import dataclass
import time

import faiss
print(f"✓ FAISS version: {faiss.__version__}")
print(f"  NumPy version: {np.__version__}")
print(f"\nPyTorch CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Model name
MODEL_NAME = "laberta_vulg"

# Retrieval configuration
K_VALUES = [5, 10, 20]  # Top-k values to retrieve

In [ ]:
# check embedding files

def check_embedding_file(filepath):
    """Check if embedding file is complete."""
    print(f"\nChecking: {filepath}")

    # File size
    file_size_mb = os.path.getsize(filepath) / (1024**2)
    print(f"  File size: {file_size_mb:.2f} MB")

    # Try to load with allow_pickle=False
    try:
        arr = np.load(filepath, allow_pickle=False)
        print(f"  ✓ Loaded successfully")
        print(f"  Shape: {arr.shape}")
        print(f"  Total elements: {arr.size:,}")
        return True
    except ValueError as e:
        print(f"  ✗ Load failed: {e}")

        # Check raw data size
        with open(filepath, 'rb') as f:
            # Skip header (128 bytes typically)
            f.seek(128)
            data = f.read()
            num_floats = len(data) // 4  # 4 bytes per float32
            print(f"  Raw data contains ~{num_floats:,} float32 values")

        return False

# Check both files
bdc_path = f"bdc_embeddings_{MODEL_NAME}.npy"
psc_path = f"psc_embeddings_{MODEL_NAME}.npy"


bdc_ok = check_embedding_file(bdc_path)
psc_ok = check_embedding_file(psc_path)

print(f"\n{'='*60}")
if bdc_ok and psc_ok:
    print("Both files are OK")
else:
    print("check embeddings")

In [ ]:
def load_embeddings(embeddings_dir: str, model_name: str) -> Tuple[np.ndarray, np.ndarray, List[str], List[str], Dict]:
    """
    Load embeddings and chunk IDs.

    Returns:
        (bdc_embeddings, psc_embeddings, bdc_ids, psc_ids, metadata)
    """
    print("Loading embeddings and IDs...\n")

    # Load embeddings
    bdc_emb_path = f"{embeddings_dir}/bdc_embeddings_{model_name}.npy"
    psc_emb_path = f"{embeddings_dir}/psc_embeddings_{model_name}.npy"

    print(f"Loading BDC embeddings from: {Path(bdc_emb_path).name}")
    bdc_embeddings = np.load(bdc_emb_path)
    print(f"  Shape: {bdc_embeddings.shape}")

    print(f"Loading PSC embeddings from: {Path(psc_emb_path).name}")
    psc_embeddings = np.load(psc_emb_path)
    print(f"  Shape: {psc_embeddings.shape}\n")

    # Load IDs from JSON
    bdc_ids_path = f"{embeddings_dir}/bdc_ids_{model_name}.json"
    psc_ids_path = f"{embeddings_dir}/psc_ids_{model_name}.json"

    print(f"Loading BDC IDs from: {Path(bdc_ids_path).name}")
    with open(bdc_ids_path, 'r') as f:
        bdc_ids = json.load(f)  # Load entire JSON array
    print(f"  Loaded {len(bdc_ids):,} IDs")

    print(f"Loading PSC IDs from: {Path(psc_ids_path).name}")
    with open(psc_ids_path, 'r') as f:
        psc_ids = json.load(f)  # Load entire JSON array
    print(f"  Loaded {len(psc_ids):,} IDs\n")

    # Load metadata
    metadata_path = f"{embeddings_dir}/metadata_{model_name}.json"
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    print(f"✓ Loaded metadata\n")

    # Verification
    assert bdc_embeddings.shape[0] == len(bdc_ids), "BDC embeddings and IDs mismatch!"
    assert psc_embeddings.shape[0] == len(psc_ids), "PSC embeddings and IDs mismatch!"
    assert bdc_embeddings.shape[1] == psc_embeddings.shape[1], "Embedding dimensions mismatch!"

    return bdc_embeddings, psc_embeddings, bdc_ids, psc_ids, metadata


# Load data
bdc_embeddings, psc_embeddings, bdc_ids, psc_ids, metadata = load_embeddings(
    DATA_DIR,
    MODEL_NAME
)

print(f"\nLoaded:")
print(f"  BDC queries: {len(bdc_ids):,}")
print(f"  PSC candidates: {len(psc_ids):,}")
print(f"  Embedding dimension: {bdc_embeddings.shape[1]}")

In [ ]:
# Get embedding dimension
embedding_dim = psc_embeddings.shape[1]

print(f"PSC embeddings shape: {psc_embeddings.shape}")
print(f"Embedding dimension: {embedding_dim}")
print()

# Build FAISS index (IndexFlatIP)
print("Building FAISS index...")
index = faiss.IndexFlatIP(embedding_dim)

# Add PSC embeddings to index
index.add(psc_embeddings.astype('float32'))

print(f"✓ FAISS index built")
print(f"  Index type: IndexFlatIP (exact inner product)")
print(f"  Total vectors: {index.ntotal:,}")
print(f"  Dimension: {embedding_dim}")

In [ ]:
# Create mappings
bdc_id_to_idx = {chunk_id: idx for idx, chunk_id in enumerate(bdc_ids)}
psc_id_to_idx = {chunk_id: idx for idx, chunk_id in enumerate(psc_ids)}

print(f"BDC mapping: {len(bdc_id_to_idx):,} queries")
print(f"PSC mapping: {len(psc_id_to_idx):,} candidates")

# Verify mappings work
sample_bdc_id = bdc_ids[0]
sample_psc_id = psc_ids[0]

print(f"\nSample BDC ID: {sample_bdc_id} → index {bdc_id_to_idx[sample_bdc_id]}")
print(f"Sample PSC ID: {sample_psc_id} → index {psc_id_to_idx[sample_psc_id]}")

In [ ]:
# Parameters
MAX_K = max(K_VALUES)

# Build retrieval results: {query_id: [(candidate_id, score), ...]}
retrieval_results = {}

print(f"Retrieving top-{MAX_K} candidates for each query...")
print(f"Total queries to process: {len(bdc_ids):,}\n")

for query_id in tqdm(bdc_ids, desc="Retrieving"):
    # Get query index
    query_idx = bdc_id_to_idx[query_id]

    # Search FAISS index
    distances, indices = index.search(
        bdc_embeddings[query_idx:query_idx+1],
        MAX_K
    )

    # Convert to (candidate_id, similarity_score) tuples
    candidates = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
        candidate_id = psc_ids[idx]
        similarity = float(dist)  # cosine similarity from normalized vectors
        candidates.append((candidate_id, similarity))

    retrieval_results[query_id] = candidates

print(f"\n✓ Retrieval complete")
print(f"  Total queries: {len(retrieval_results):,}")
print(f"  Candidates per query: {MAX_K}")

# Verify structure
sample_query = list(retrieval_results.keys())[0]
print(f"\nSample retrieval result:")
print(f"  Query ID: {sample_query}")
print(f"  Top 3 candidates:")
for cand_id, score in retrieval_results[sample_query][:3]:
    print(f"    - {cand_id}: {score:.4f}")

In [ ]:
# Create results for each k value (for compatibility with evaluation)
results_by_k = {}

for k in K_VALUES:
    results_by_k[k] = {
        query_id: candidates[:k]
        for query_id, candidates in retrieval_results.items()
    }

# Save retrieval results
output_data = {
    "metadata": {
        "model_name": MODEL_NAME,
        "total_queries": len(retrieval_results),
        "total_candidates": len(psc_ids),
        "max_k": MAX_K,
        "k_values": K_VALUES,
        "index_type": "FAISS IndexFlatIP",
    },
    "retrieval_results": {
        query_id: [
            {
                "candidate_id": cand_id,
                "similarity": score,
                "rank": rank
            }
            for rank, (cand_id, score) in enumerate(candidates, 1)
        ]
        for query_id, candidates in retrieval_results.items()
    }
}

# Convert OUTPUT_DIR to Path if it's a string
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Save JSON
output_path = output_dir / f"retrieval_results_{MODEL_NAME.lower()}.json"
with open(output_path, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"Saved full retrieval results to: {output_path}")

#  save PKL
import pickle
pickle_path = output_dir / f"retrieval_results_{MODEL_NAME.lower()}.pkl"
with open(pickle_path, 'wb') as f:
    pickle.dump(retrieval_results, f)

print(f"\nFiles saved:")
print(f"  - JSON: {output_path}")
print(f"  - Pickle: {pickle_path}")

In [ ]:
# Convert OUTPUT_DIR to Path
output_dir = Path(OUTPUT_DIR)

retrieval_metadata = {
    "model_name": MODEL_NAME,
    "retrieval_config": {
        "k_values": K_VALUES,
        "max_k": MAX_K,
        "index_type": "IndexFlatIP"
    },
    "statistics": {
        "num_queries": len(bdc_ids),
        "num_candidates": len(psc_ids),
        "embedding_dim": bdc_embeddings.shape[1],
        "total_retrievals": len(retrieval_results)
    },
    "results_per_k": {}
}

# Add statistics for each k
for k in K_VALUES:
    # Get top-k results for all queries
    topk_results = [candidates[:k] for candidates in retrieval_results.values()]

    # Calculate statistics
    avg_similarity = np.mean([
        candidates[0][1] if candidates else 0.0  # Top-1 similarity
        for candidates in topk_results
    ])

    retrieval_metadata["results_per_k"][f"k{k}"] = {
        "k": k,
        "num_queries": len(retrieval_results),
        "candidates_per_query": k,
        "avg_top1_similarity": float(avg_similarity),
        "output_file": f"retrieval_results_{MODEL_NAME.lower()}.json"
    }

# Save metadata
metadata_path = output_dir / f"retrieval_metadata_{MODEL_NAME.lower()}.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(retrieval_metadata, f, indent=2, ensure_ascii=False)

print(f"Saved metadata to: {metadata_path}")